In [ ]:
# Cell 1 - Clone repository
%cd /content
!if [ ! -d VDT_GD2 ]; then git clone https://github.com/Darkin72/VDT_GD2.git; fi
%cd /content/VDT_GD2
!git pull


In [ ]:
# Cell 2 - Paper training configuration
DEVICE = "cuda"
WEIGHT_DECAY = 1e-4
MIN_LR = 1e-6
PATIENCE = 50
MULTI_SCALE = "64, 128, 256"
AUGMENTATION = "flip H/V, rotation 0/90/180/270, brightness/contrast +/-0.1"

CONFIG = {
    # Paper: synthetic datasets use 1000 epochs, batch 16 and cosine annealing.
    "SOTS_ITS": {"epochs": 1000, "batch": 16, "lr": 2e-4, "scheduler": "cosine"},
    "SOTS_OTS": {"epochs": 1000, "batch": 16, "lr": 1e-4, "scheduler": "cosine"},
    # I-HAZE/O-HAZY are real-world datasets; use the paper's real-world settings.
    "I_HAZE": {"epochs": 500, "batch": 8, "lr": 2e-4, "scheduler": "step"},
    "O_HAZY": {"epochs": 500, "batch": 8, "lr": 2e-4, "scheduler": "step"},
}
for dataset, cfg in CONFIG.items():
    print(dataset, cfg)
print("device:", DEVICE, "| weight_decay:", WEIGHT_DECAY, "| min_lr:", MIN_LR)
print("multi-scale:", MULTI_SCALE, "| augmentation:", AUGMENTATION)

EVAL_MAX_SIDE = 1024  # Match Bench.ipynb-style bounded full-image evaluation


In [ ]:
# Cell 3 - Train with the paper settings
!pip -q install pillow matplotlib numpy
%cd /content/VDT_GD2
import os
os.environ["HW_DEVICE"] = DEVICE
os.environ["HW_WD"] = str(WEIGHT_DECAY)
os.environ["HW_MIN_LR"] = str(MIN_LR)
os.environ["HW_PATIENCE"] = str(PATIENCE)

c = CONFIG["I_HAZE"]
os.environ["HW_EPOCHS"] = str(c["epochs"]); os.environ["HW_BATCH"] = str(c["batch"]); os.environ["HW_LR"] = str(c["lr"]); os.environ["HW_SCHEDULER"] = c["scheduler"]
!python -m solution.wavelet_dehaze.train dataset/I-HAZE/train --val-data dataset/I-HAZE/val --device "$HW_DEVICE" --epochs "$HW_EPOCHS" --batch-size "$HW_BATCH" --size 256 --lr "$HW_LR" --weight-decay "$HW_WD" --scheduler "$HW_SCHEDULER" --step-size 100 --min-lr "$HW_MIN_LR" --augment --multi-scale --patience "$HW_PATIENCE" --out haze_wavelet_i_haze.pt --history history_i_haze.json

c = CONFIG["O_HAZY"]
os.environ["HW_EPOCHS"] = str(c["epochs"]); os.environ["HW_BATCH"] = str(c["batch"]); os.environ["HW_LR"] = str(c["lr"]); os.environ["HW_SCHEDULER"] = c["scheduler"]
!python -m solution.wavelet_dehaze.train dataset/O-HAZY/train --val-data dataset/O-HAZY/val --device "$HW_DEVICE" --epochs "$HW_EPOCHS" --batch-size "$HW_BATCH" --size 256 --lr "$HW_LR" --weight-decay "$HW_WD" --scheduler "$HW_SCHEDULER" --step-size 100 --min-lr "$HW_MIN_LR" --augment --multi-scale --patience "$HW_PATIENCE" --out haze_wavelet_o_hazy.pt --history history_o_hazy.json

SOTS_ROOT = "dataset/Synthetic Objective Testing Set (SOTS) [RESIDE]"

c = CONFIG["SOTS_ITS"]
os.environ["HW_EPOCHS"] = str(c["epochs"]); os.environ["HW_BATCH"] = str(c["batch"]); os.environ["HW_LR"] = str(c["lr"]); os.environ["HW_SCHEDULER"] = c["scheduler"]
!python -m solution.wavelet_dehaze.train "$SOTS_ROOT/indoor/train" --val-data "$SOTS_ROOT/indoor/val" --device "$HW_DEVICE" --epochs "$HW_EPOCHS" --batch-size "$HW_BATCH" --size 256 --lr "$HW_LR" --weight-decay "$HW_WD" --scheduler "$HW_SCHEDULER" --min-lr "$HW_MIN_LR" --augment --multi-scale --patience "$HW_PATIENCE" --out haze_wavelet_sots_its.pt --history history_sots_its.json

c = CONFIG["SOTS_OTS"]
os.environ["HW_EPOCHS"] = str(c["epochs"]); os.environ["HW_BATCH"] = str(c["batch"]); os.environ["HW_LR"] = str(c["lr"]); os.environ["HW_SCHEDULER"] = c["scheduler"]
!python -m solution.wavelet_dehaze.train "$SOTS_ROOT/outdoor/train" --val-data "$SOTS_ROOT/outdoor/val" --device "$HW_DEVICE" --epochs "$HW_EPOCHS" --batch-size "$HW_BATCH" --size 256 --lr "$HW_LR" --weight-decay "$HW_WD" --scheduler "$HW_SCHEDULER" --min-lr "$HW_MIN_LR" --augment --multi-scale --patience "$HW_PATIENCE" --out haze_wavelet_sots_ots.pt --history history_sots_ots.json


In [ ]:
# Cell 4 - Preview metrics separately for each dataset
import json
from pathlib import Path
import matplotlib.pyplot as plt

files = {
    "I-HAZE": "history_i_haze.json",
    "O-HAZY": "history_o_hazy.json",
    "SOTS-ITS": "history_sots_its.json",
    "SOTS-OTS": "history_sots_ots.json",
}
histories = {
    name: json.loads(Path(path).read_text())
    for name, path in files.items()
    if Path(path).exists()
}

if not histories:
    raise FileNotFoundError("No history_*.json files found. Run Cell 3 first.")

fig, axes = plt.subplots(
    len(histories), 3,
    figsize=(20, 5 * len(histories)),
    squeeze=False,
)
for row, (name, records) in enumerate(histories.items()):
    epoch = [item["epoch"] for item in records]
    metrics = [
        ("train_loss", "val_loss", "Loss"),
        ("train_psnr", "val_psnr", "PSNR (dB)"),
        ("train_ssim", "val_ssim", "SSIM"),
    ]
    for col, (train_key, val_key, title) in enumerate(metrics):
        ax = axes[row, col]
        ax.plot(epoch, [item[train_key] for item in records], label="Train", linewidth=2)
        ax.plot(epoch, [item[val_key] for item in records], "--", label="Validation", linewidth=2)
        ax.set_title(f"{name} - {title}")
        ax.set_xlabel("Epoch")
        ax.set_ylabel(title)
        ax.grid(alpha=0.25)
        ax.legend()

plt.tight_layout()
plt.savefig("haze_wavelet_training_metrics_by_dataset.png", dpi=160)
plt.show()

In [ ]:
# Cell 5 - Benchmark: 4 Excel reports (dataset, fog, domain gap, hardware)
from pathlib import Path
!pip -q install openpyxl psutil scikit-image
%cd /content/VDT_GD2

RESULT_DIR = "utils/evaluation_results/hazewavenet"
import os
os.environ["HW_DEVICE"] = DEVICE
os.environ["HW_MAX_SIDE"] = str(EVAL_MAX_SIDE)
os.environ["HW_RESULT_DIR"] = RESULT_DIR
!mkdir -p "$HW_RESULT_DIR"

!python -m solution.wavelet_dehaze.evaluate   --data-root dataset   --output-dir "$HW_RESULT_DIR"   --checkpoint-i-haze haze_wavelet_i_haze.pt   --checkpoint-o-hazy haze_wavelet_o_hazy.pt   --checkpoint-sots-its haze_wavelet_sots_its.pt   --checkpoint-sots-ots haze_wavelet_sots_ots.pt   --device "$HW_DEVICE"   --max-side "$HW_MAX_SIDE"

print("Excel reports:")
for path in sorted(Path(RESULT_DIR).glob("*.xlsx")):
    print(" -", path)
